# Introduction to Arabic Speech Technologies
## Chapter 9 notebook: Diacritization, grapheme to phoneme, and comparing two front ends

Electronic supplementary material for *Introduction to Arabic Speech Technologies* by Hend S. Al-Khalifa.

Chapter 9 asks the reader to send undiacritized Arabic news text through two open-source diacritizers, convert each output to a phoneme sequence as in this notebook, and describe where the two disagree and what those differences would do to a synthesiser. The notebook supplies the scoring machinery: a diacritic error rate, the grapheme-to-phoneme conversion, a disagreement report, and hooks where a real diacritizer can be plugged in.

**Contents**

1. Why the front end has to decide: one spelling, several readings
2. Grapheme to phoneme for diacritized text
3. Diacritic error rate and word error rate on vocalised forms
4. Comparing two diacritizers, with a disagreement report
5. What the disagreements do to synthesis
6. Optional: running a real diacritizer

**Running it.** The notebook needs only `numpy`, `scipy` and `matplotlib` (see `requirements.txt`). It runs top to bottom with no downloads and no audio files: where a recording is useful, the notebook synthesises one, and a cell is provided for reading your own WAV file instead. Optional cells that need extra packages or internet access are marked *Optional*.


## 1. One spelling, several readings

Ordinary Arabic leaves the short vowels unwritten, so the text a synthesiser receives underdetermines its own pronunciation. The front end has to choose, and a wrong choice is audible.

In [1]:
readings = {
 "علم": [("عِلْم", "knowledge"), ("عَلَم", "flag"), ("عَلِمَ", "he knew"), ("عَلَّمَ", "he taught")],
 "كتب": [("كَتَبَ", "he wrote"), ("كُتُب", "books"), ("كَتَّبَ", "he made someone write")],
 "ذهب": [("ذَهَبَ", "he went"), ("ذَهَب", "gold")],
}
for bare, options in readings.items():
    print(f"{bare}: " + "   ".join(f"{v} ({g})" for v, g in options))

علم: عِلْم (knowledge)   عَلَم (flag)   عَلِمَ (he knew)   عَلَّمَ (he taught)
كتب: كَتَبَ (he wrote)   كُتُب (books)   كَتَّبَ (he made someone write)
ذهب: ذَهَبَ (he went)   ذَهَب (gold)


## 2. Grapheme to phoneme

The converter is the one built for Chapter 2, repeated here so that this notebook stands alone: consonants map to phones, a shadda doubles the consonant, a short vowel followed by its matching letter is a long vowel, and the definite article assimilates before a sun letter.

In [2]:
CONSONANTS = {
 "ء":"ʔ","أ":"ʔ","إ":"ʔ","آ":"ʔ","ب":"b","ت":"t","ث":"θ","ج":"dʒ","ح":"ħ","خ":"x","د":"d","ذ":"ð","ر":"r","ز":"z",
 "س":"s","ش":"ʃ","ص":"sˤ","ض":"dˤ","ط":"tˤ","ظ":"ðˤ","ع":"ʕ","غ":"ɣ","ف":"f","ق":"q","ك":"k","ل":"l","م":"m",
 "ن":"n","ه":"h","و":"w","ي":"j",
}
SHORT = {"َ":"a", "ِ":"i", "ُ":"u"}
TANWIN = {"ً":"an", "ٍ":"in", "ٌ":"un"}
SHADDA, SUKUN = "ّ", "ْ"
SUN = set("تثدذرزسشصضطظلن")

def canonical(word):
    out = []
    for ch in word:
        if ch == SHADDA and out and out[-1] in SHORT:
            v = out.pop(); out.append(SHADDA); out.append(v)
        else:
            out.append(ch)
    return "".join(out)

def g2p(word):
    chars, out, i = list(canonical(word)), [], 0
    while i < len(chars):
        ch = chars[i]
        if ch in CONSONANTS:
            phone = CONSONANTS[ch]
            if i + 1 < len(chars) and chars[i + 1] == SHADDA:
                out += [phone, phone]; i += 2; continue
            out.append(phone); i += 1; continue
        if ch in SHORT:
            v = ch and SHORT[ch]
            nxt = chars[i + 1] if i + 1 < len(chars) else ""
            if (v, nxt) in {("a", "ا"), ("u", "و"), ("i", "ي"), ("a", "ى")}:
                out.append(v + "ː"); i += 2
            else:
                out.append(v); i += 1
            continue
        if ch in TANWIN: out += list(TANWIN[ch]); i += 1; continue
        if ch == SUKUN: i += 1; continue
        if ch == "ا": out.append("aː"); i += 1; continue
        if ch == "ى": out.append("aː"); i += 1; continue
        if ch == "ة":
            if not out or out[-1] != "a": out.append("a")
            i += 1; continue
        i += 1
    return out

def g2p_sentence(text):
    words = []
    for w in text.split():
        if w.startswith("ال") and len(w) > 2:
            rest = w[2:].lstrip("ًٌٍَُِّْ")      # ignore a stray mark on the article
            first = next((c for c in rest if c in CONSONANTS), "")
            ph = g2p(rest)
            if first in SUN:
                doubled = len(ph) > 1 and ph[0] == ph[1]
                words.append(["ʔ", "a"] + (ph if doubled else [ph[0]] + ph))
            else:
                words.append(["ʔ", "a", "l"] + ph)
        else:
            words.append(g2p(w))
    return words

def phones(word):
    # Phone sequence for one word, with the definite article handled.
    return g2p_sentence(word)[0]

for v, gloss in [("عِلْم", "knowledge"), ("عَلَّمَ", "he taught"), ("الشَّمْس", "the sun"), ("القَمَر", "the moon")]:
    print(f"{v:10} /{' '.join(g2p_sentence(v)[0]):18}/  {gloss}")

عِلْم      /ʕ i l m           /  knowledge
عَلَّمَ    /ʕ a l l a m a     /  he taught
الشَّمْس   /ʔ a ʃ ʃ a m s     /  the sun
القَمَر    /ʔ a l q a m a r   /  the moon


## 3. Scoring a diacritizer

Two numbers are usual. The **diacritic error rate** compares the mark on every letter, and is normally reported both with and without the final letter of each word, because case and mood endings (iʿrāb) are the hardest part and are often excluded. The **word error rate on vocalised forms** counts a word as wrong if any of its marks are wrong.

In [3]:
MARKS = set("ًٌٍَُِّْ")

def strip_diacritics(text):
    return "".join(c for c in text if c not in MARKS)

def letters_with_marks(word):
    pairs, i = [], 0
    chars = list(word)
    while i < len(chars):
        if chars[i] in MARKS: i += 1; continue
        mark = ""
        j = i + 1
        while j < len(chars) and chars[j] in MARKS:
            mark += chars[j]; j += 1
        pairs.append((chars[i], mark)); i = j
    return pairs

def diacritic_error_rate(ref, hyp, ignore_last_letter=False):
    wrong = total = 0
    for r_word, h_word in zip(ref.split(), hyp.split()):
        r_pairs, h_pairs = letters_with_marks(r_word), letters_with_marks(h_word)
        if strip_diacritics(r_word) != strip_diacritics(h_word):
            continue                      # different letters: not a diacritization difference
        pairs = zip(r_pairs, h_pairs)
        n = len(r_pairs) - (1 if ignore_last_letter and len(r_pairs) > 1 else 0)
        for k, ((rl, rm), (hl, hm)) in enumerate(pairs):
            if k >= n: break
            total += 1
            wrong += int(rm != hm)
    return wrong / max(total, 1)

def word_error_rate_vocalised(ref, hyp):
    r, h = ref.split(), hyp.split()
    wrong = sum(1 for a, b in zip(r, h) if a != b) + abs(len(r) - len(h))
    return wrong / max(len(r), 1)

gold = "ذَهَبَ الأَوْلادُ إِلَى المَدْرَسَةِ"
sys1 = "ذَهَبَ الأَوْلادُ إِلَى المَدْرَسَةِ"
sys2 = "ذَهَبُ الأَوْلادِ إِلَى المُدَرِّسَةِ"
for name, hyp in (("system 1", sys1), ("system 2", sys2)):
    print(f"{name}: DER {diacritic_error_rate(gold, hyp):.3f}   "
          f"DER without final letter {diacritic_error_rate(gold, hyp, True):.3f}   "
          f"word error rate {word_error_rate_vocalised(gold, hyp):.3f}")

system 1: DER 0.000   DER without final letter 0.000   word error rate 0.000
system 2: DER 0.250   DER without final letter 0.188   word error rate 0.750


## 4. Comparing two diacritizers

The function below takes the same undiacritized sentences through two diacritizers, converts both outputs to phonemes, and prints every word where the two disagree together with the phonemic consequence. Here the two systems are stand-ins: a lookup table of gold forms and a rule-based baseline that guesses the most common vowel pattern. Replace either with a real system in section 6.

In [4]:
GOLD = {
 "ذهب": "ذَهَبَ", "الأولاد": "الأَوْلادُ", "إلى": "إِلَى", "المدرسة": "المَدْرَسَةِ",
 "علم": "عِلْم", "الطالب": "الطّالِبُ", "كتب": "كَتَبَ", "الدرس": "الدَّرْسَ",
}
def diacritizer_a(word):                    # a lookup-table system
    return GOLD.get(word, word)

def diacritizer_b(word):                    # a crude rule: fatha on every consonant, damma at the end
    out = ""
    letters = [c for c in word]
    for k, c in enumerate(letters):
        out += c
        if c in CONSONANTS and k < len(letters) - 1 and letters[k + 1] not in "اوي":
            out += "َ"
    return out + ("ُ" if word[-1] in CONSONANTS else "")

sentences = ["ذهب الأولاد إلى المدرسة", "كتب الطالب الدرس"]
for s in sentences:
    a = " ".join(diacritizer_a(w) for w in s.split())
    b = " ".join(diacritizer_b(w) for w in s.split())
    print(f"input : {s}")
    print(f"sys A : {a}")
    print(f"sys B : {b}")
    print(f"        DER between the two systems: {diacritic_error_rate(a, b):.3f}")
    for w, wa, wb in zip(s.split(), a.split(), b.split()):
        if wa != wb:
            print(f"        {w:10} A {wa:12} /{' '.join(phones(wa)):18}/   B {wb:12} /{' '.join(phones(wb)):18}/")
    print()

input : ذهب الأولاد إلى المدرسة
sys A : ذَهَبَ الأَوْلادُ إِلَى المَدْرَسَةِ
sys B : ذَهَبُ الَأوَلادُ إَلَى الَمَدَرَسَة
        DER between the two systems: 0.400
        ذهب        A ذَهَبَ       /ð a h a b a       /   B ذَهَبُ       /ð a h a b u       /
        الأولاد    A الأَوْلادُ   /ʔ a l ʔ a w l aː d u/   B الَأوَلادُ   /ʔ a l ʔ w a l aː d u/
        إلى        A إِلَى        /ʔ i l aː          /   B إَلَى        /ʔ a l aː          /
        المدرسة    A المَدْرَسَةِ /ʔ a l m a d r a s a i/   B الَمَدَرَسَة /ʔ a l m a d a r a s a/

input : كتب الطالب الدرس
sys A : كَتَبَ الطّالِبُ الدَّرْسَ
sys B : كَتَبُ الَطالَبُ الَدَرَسُ
        DER between the two systems: 0.571
        كتب        A كَتَبَ       /k a t a b a       /   B كَتَبُ       /k a t a b u       /
        الطالب     A الطّالِبُ    /ʔ a tˤ tˤ aː l i b u/   B الَطالَبُ    /ʔ a tˤ tˤ aː l a b u/
        الدرس      A الدَّرْسَ    /ʔ a d d a r s a   /   B الَدَرَسُ    /ʔ a d d a r a s u /



## 5. What the disagreements do to synthesis

A disagreement matters when it changes the phoneme string, and matters more when it changes a vowel's length or a consonant's gemination, because those carry meaning. The count below separates the two cases.

In [5]:
def phoneme_diff(word_a, word_b):
    pa, pb = phones(word_a), phones(word_b)
    if pa == pb: return "identical"
    if [p.replace("ː", "") for p in pa] == [p.replace("ː", "") for p in pb]: return "vowel length only"
    if len(pa) != len(pb): return "gemination or segment count"
    return "segment quality"

counts = {}
for s in sentences:
    for w in s.split():
        kind = phoneme_diff(diacritizer_a(w), diacritizer_b(w))
        counts[kind] = counts.get(kind, 0) + 1
for kind, n in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"{kind:28} {n}")
print("\nA synthesiser reads the phoneme string, so anything other than 'identical' is audible.")

segment quality              6
gemination or segment count  1

A synthesiser reads the phoneme string, so anything other than 'identical' is audible.


## 6. Optional: running a real diacritizer

The exercise in Chapter 9 asks for two open-source systems. Two common choices are CAMeL Tools and Mishkal. Both need installation and, for CAMeL Tools, a model download, so the cell is left commented out. Wrap either one in a function with the same signature as `diacritizer_a` and the rest of the notebook works unchanged.

In [6]:
# !pip install camel-tools
# from camel_tools.disambig.mle import MLEDisambiguator
# mle = MLEDisambiguator.pretrained()
# def diacritizer_camel(word):
#     analysis = mle.disambiguate([word])[0].analyses
#     return analysis[0].analysis["diac"] if analysis else word
#
# !pip install mishkal
# from mishkal.tashkeel import TashkeelClass
# vocalizer = TashkeelClass()
# def diacritizer_mishkal(word):
#     return vocalizer.tashkeel(word)
#
# Then: compare them on a news paragraph of your own, report DER with and without
# the final letter, and describe which disagreements change the phoneme string.